# Engineering workflows

Pull requests, alerts, and reviews from the fixtures. The risk number is a weighted sum in Python. Change a weight, not a prompt, when the mix is wrong.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 28. Should this pull request auto-advance?

Four separate questions: sensitive files, an honest description, scope, and tests. Code combines them.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "touches_sensitive": Noul(instructions="Does `diff_summary` modify auth, secrets, CI config, or delete existing code?"),
        "description_matches": Noul(instructions="Does `body` accurately describe `diff_summary`?"),
        "scope": Score(
            instructions="How many independent changes does this pull request bundle?",
            criteria=["One focused change", "One change plus a small tweak", "Several unrelated changes"],
        ),
        "tests_added": Noul(instructions="Does `diff_summary` mention tests for the change?"),
    }
    for pull in load_json("prs.json"):
        response = ask(pull, questions)
        show(response)
        risk = (
            0.45 * response.nouls["touches_sensitive"].noul
            + 0.25 * (1 - response.nouls["description_matches"].noul)
            + 0.15 * response.scores["scope"].score / 2
            + 0.15 * (1 - response.nouls["tests_added"].noul)
        )
        route = "human_review" if risk >= 0.55 or response.scores["scope"].confidence < 0.45 else "auto_advance"
        print(pull["id"], "risk", round(risk, 2), "->", route)


**What you should see.** PR-18, the single retry with a test, should be safer than PR-19, which touches auth and CI and has no tests.


## 29. Triage an alert against open incidents

A new checkout 500 is the same story as open incident INC-14. A printer jam is not a customer outage.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "customer_impact": Score(
            instructions="How much are shoppers affected?",
            criteria=["None", "Degraded", "Down"],
        ),
        "likely_cause": Choice(
            instructions="Most likely cause",
            criteria={
                "deploy": "A recent change",
                "dependency": "An outside service",
                "capacity": "Load or a full disk",
                "unknown": "Not enough to say",
            },
        ),
        "duplicate_of_open": Noul(instructions="Does `alert` describe the same issue as an item in `open_incidents`?"),
    }
    open_items = open_incidents()
    for alert in load_json("alerts.json"):
        response = ask({"alert": alert["text"], "open_incidents": open_items}, questions)
        show(response)
        if response.nouls["duplicate_of_open"].noul > 0.65:
            route = "attach_to_existing"
        elif response.scores["customer_impact"].score > 1.5:
            route = "SEV1"
        elif response.scores["customer_impact"].score > 0.5:
            route = "SEV2"
        else:
            route = "SEV3"
        print(alert["id"], "->", route)


**What you should see.** The checkout alert should attach to the existing incident or open as a high severity. The printer jam should be SEV3 or another low bucket.


## 30. Turn reviews into numbers

Each Noul is a feature. This cell prints the numbers and a weighted score. A classical model could fit them later. We do not train one here.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    features = {
        "mentions_price": "Does the review mention price or value?",
        "mentions_defect": "Does the review describe something broken?",
        "would_recommend": "Does the reviewer say they would buy or recommend it?",
    }
    questions = {name: Noul(instructions=text) for name, text in features.items()}
    requests = [{"state": review, "questions": questions} for review in load_json("content.json")["reviews"]]
    for review, response in zip(load_json("content.json")["reviews"], ask_many(requests)):
        show(response)
        score = (
            0.2 * response.nouls["mentions_price"].noul
            + 0.5 * response.nouls["mentions_defect"].noul
            + 0.3 * (1 - response.nouls["would_recommend"].noul)
        )
        print(round(score, 2), review)


**What you should see.** The broken zipper should score higher (worse) than the sturdy bottle. These are features, not a trained classifier.
